<a href="https://colab.research.google.com/github/id-21/rgb-x/blob/normal-irradiance-pipeline/RGB_X_Colab_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RGB→X Model: Normal Maps & Irradiance (Shadow) Extraction

**Purpose:** Extract normal maps and irradiance (shadow) layers from room photos for ST-RoomNet V3 wallpaper visualization.

**Features:**
- Custom image upload with parameter controls
- Dynamic sample image processing (N preloaded room photos)
- Individual hierarchical ZIP downloads per image
- Batch processing: "Process All Samples" button
- Master ZIP download: "Download All" (excludes failed runs)
- Editable callbacks: Modify processing logic without reloading model

**Model stays loaded:** Run Cell 4 once, then edit Cells 7-10 and re-run Gradio without model reload.

---
## Cell 1: Environment Setup (Python 3.9 venv)
**Run this first:** Installs Python 3.9 and dependencies in isolated environment.

In [1]:
%%bash
# Install Python 3.9 (Google Colab defaults to Python 3.12)
apt-get update
apt-get install -y python3.9 python3.9-venv python3.9-dev

# Create isolated Python 3.9 virtual environment
python3.9 -m venv /content/py39env

# Activate venv and install dependencies
source /content/py39env/bin/activate
pip install --upgrade pip
pip install torch==2.3.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install diffusers==0.20.0 gradio==3.48.0 opencv-python==4.9.0.80 transformers==4.40.2 huggingface-hub==0.25.0 imageio==2.34.1 numpy==1.26.4

echo "✓ Python 3.9 environment created at /content/py39env"

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,080 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,809 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
# Verify GPU and clear conflicting environment variables
import os
import sys

# Clear Colab's Jupyter-specific matplotlib backend (causes conflicts in standalone scripts)
os.environ.pop('MPLBACKEND', None)

# Verify Python version
# print(f"Python version: {sys.version}")
# assert sys.version_info[:2] == (3, 9), "ERROR: Python 3.9 required! Re-run Cell 1."
# print("Python executable:", sys.executable)
# assert "py39env" in sys.executable, "ERROR: Notebook not running inside py39env. Please restart runtime after installation."
!{os.path.join('/content/py39env/bin/python')} -V  # version inside py39env

# Verify GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    print("WARNING: GPU not detected. Ensure Colab runtime is set to GPU (Runtime → Change runtime type → T4 GPU)")

print("\n✓ Environment ready!")

Python 3.9.24
CUDA available: True
GPU: Tesla T4
CUDA version: 12.6

✓ Environment ready!


---
## Cell 2: Clone Repository and Navigate to Working Directory
**Note:** Replace `<REPO_URL>` with your repository URL (or skip if already cloned).

In [9]:
# Clone YOUR private repo (includes shadow download functionality)
# This requires a Personal Access Token for private repo access

from getpass import getpass
import os

print("🔐 GitHub Authentication Required")
print("Generate token at: https://github.com/settings/tokens")
print("Permissions needed: 'repo' (full control)")
print()

token = getpass('Enter your GitHub Personal Access Token: ')

# Clone using token
!git clone https://{token}@github.com/id-21/rgb-x.git

# Clear token from memory for security
token = None

%cd /content/rgb-x/rgbx
# %git checkout normal-irradiance-pipeline
# %echo "✓ Repository ready at: $(pwd)"

🔐 GitHub Authentication Required
Generate token at: https://github.com/settings/tokens
Permissions needed: 'repo' (full control)

Enter your GitHub Personal Access Token: ··········
Cloning into 'rgb-x'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 134 (delta 48), reused 119 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 15.59 MiB | 15.95 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/rgb-x/rgbx


UsageError: Line magic function `%echo` not found.


In [11]:
%%bash
# Clone repository (skip if already cloned)
if [ ! -d "/content/rgb-x" ]; then
    echo "Cloning repository..."
    # For public repos:
    # git clone https://github.com/your-username/rgb-x.git /content/rgb-x

    # For private repos, use GitHub token:
    git clone https://[token]@github.com/id-21/rgb-x.git/content/rgb-x

    echo "ERROR: Update this cell with your repo URL and uncomment the git clone line!"
    exit 1
else
    echo "Repository already exists at /content/rgb-x"
fi

# Checkout correct branch
cd /content/rgb-x
git checkout normal-irradiance-pipeline

echo "✓ Repository ready at: $(pwd)"

Repository already exists at /content/rgb-x
Your branch is up to date with 'origin/normal-irradiance-pipeline'.
✓ Repository ready at: /content/rgb-x


Already on 'normal-irradiance-pipeline'


In [12]:
# Verify repository structure and navigate to working directory
import os

os.chdir('/content/rgb-x/normal_irradiance')
print(f"Working directory: {os.getcwd()}")

# Verify required files exist
required_files = ['pipeline_normal_irradiance.py', 'load_image.py', 'sample_images/']
for file in required_files:
    if os.path.exists(file):
        print(f"✓ Found: {file}")
    else:
        print(f"✗ Missing: {file} - Check repository structure!")

print("\n✓ Ready to import modules!")

Working directory: /content/rgb-x/normal_irradiance
✓ Found: pipeline_normal_irradiance.py
✓ Found: load_image.py
✓ Found: sample_images/

✓ Ready to import modules!


---
## Cell 3: Import Required Modules
**Run once:** Imports all dependencies for pipeline and Gradio interface.

In [13]:
# Enable OpenEXR support in OpenCV
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

# Core imports
import gradio as gr
import torch
import torchvision
import cv2
import numpy as np
import zipfile
import glob as glob_module
from diffusers import DDIMScheduler
from PIL import Image

# Project-specific imports
from pipeline_normal_irradiance import StableDiffusionAOVMatEstPipeline
from load_image import load_ldr_image, load_exr_image

print("✓ All modules imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Gradio version: {gr.__version__}")

✓ All modules imported successfully!
PyTorch version: 2.8.0+cu126
Gradio version: 5.49.0


---
## Cell 4: Initialize Pipeline (Run Once)
**IMPORTANT:** Model stays loaded in GPU memory. Edit cells below without re-running this cell.

In [14]:
# Load RGB→X pipeline from HuggingFace
print("Loading model from HuggingFace (zheng95z/rgb-to-x)...")
print("This may take 5-10 seconds on first run (downloads ~4GB model)")

pipe = StableDiffusionAOVMatEstPipeline.from_pretrained(
    "zheng95z/rgb-to-x",
    torch_dtype=torch.float16,
    cache_dir="model_cache_colab"
)

# Configure DDIM scheduler with zero-SNR rescaling
pipe.scheduler = DDIMScheduler.from_config(
    pipe.scheduler.config,
    rescale_betas_zero_snr=True,
    timestep_spacing="trailing"
)

# Disable progress bar (cleaner output in notebooks)
pipe.set_progress_bar_config(disable=True)

# Move to GPU
pipe.to("cuda")

print("\n" + "="*60)
print("✓ Pipeline loaded successfully!")
print("Model is now in GPU memory and will stay loaded.")
print("You can edit Gradio cells (7-10) and re-run without reloading model.")
print("="*60)

Loading model from HuggingFace (zheng95z/rgb-to-x)...
This may take 5-10 seconds on first run (downloads ~4GB model)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/442 [00:00<?, ?B/s]

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

... (8 outputs left out)

config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


TypeError: CLIPTextModel.__init__() got an unexpected keyword argument 'offload_state_dict'

---
## Cell 5: ZIP Helper Function
**Editable:** Creates hierarchical ZIP structure with renamed files.

In [15]:
def create_hierarchical_zip(input_path, aov_files, output_zip_name):
    """
    Create hierarchical ZIP with input image and AOV outputs.

    Args:
        input_path: Path to original input image
        aov_files: Dictionary of {aov_name: {file_type: path}}
        output_zip_name: Name for output ZIP file

    Returns:
        Path to created ZIP file

    ZIP Structure:
        input_images/
            original_filename.jpg
        normal_model/
            filename_normal_npy.npy
            filename_normal_preview.png
        irradiance_model/
            filename_irradiance_npy.npy
            filename_irradiance.exr
            filename_irradiance_preview.png
    """
    import os

    # Get filename without extension
    input_filename = os.path.basename(input_path)
    filename_base = os.path.splitext(input_filename)[0]

    # Create ZIP in /tmp directory
    zip_path = f"/tmp/{output_zip_name}"

    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add input image
        zipf.write(input_path, arcname=f"input_images/{input_filename}")

        # Add AOV files with hierarchical structure
        for aov_name, file_dict in aov_files.items():
            for file_type, file_path in file_dict.items():
                if file_path and os.path.exists(file_path):
                    # Determine folder and create appropriate filename
                    if aov_name == "normal":
                        folder = "normal_model"
                        if file_type == "npy":
                            arc_name = f"{folder}/{filename_base}_normal_npy.npy"
                        else:  # png
                            arc_name = f"{folder}/{filename_base}_normal_preview.png"

                    elif aov_name == "irradiance":
                        folder = "irradiance_model"
                        if file_type == "npy":
                            arc_name = f"{folder}/{filename_base}_irradiance_npy.npy"
                        elif file_type == "exr":
                            arc_name = f"{folder}/{filename_base}_irradiance.exr"
                        else:  # png
                            arc_name = f"{folder}/{filename_base}_irradiance_preview.png"

                    zipf.write(file_path, arcname=arc_name)

    return zip_path

print("✓ ZIP helper function defined")

✓ ZIP helper function defined


---
## Cell 6: AOV Save Helper Function
**Editable:** Saves normal maps and irradiance with correct gamma handling.

In [16]:
def save_aov_images(aov_dict, input_filename, output_dir="/tmp/rgb2x_output"):
    """
    Save AOV images as NPY/EXR and PNG for download.

    Args:
        aov_dict: Dictionary of {aov_name: numpy_array} where numpy_array is (H, W, C) float32
        input_filename: Original filename for naming output files
        output_dir: Directory to save files

    Returns:
        Dictionary of {aov_name: {file_type: path}}
    """
    os.makedirs(output_dir, exist_ok=True)
    saved_files = {}

    # Get base filename without extension
    filename_base = os.path.splitext(os.path.basename(input_filename))[0]

    for aov_name, img_np in aov_dict.items():
        # img_np is already a numpy array in (H, W, C) format from pipeline output_type="np"
        saved_files[aov_name] = {}

        # Save normal maps as NPY and PNG
        if aov_name == "normal":
            # Save as .npy file (raw numpy array in linear space)
            npy_path = os.path.join(output_dir, f"{filename_base}_normal_npy.npy")
            np.save(npy_path, img_np.astype(np.float32))
            saved_files[aov_name]["npy"] = npy_path

            # Save PNG preview (8-bit, display-ready)
            png_path = os.path.join(output_dir, f"{filename_base}_normal_preview.png")
            img_png = (np.clip(img_np, 0, 1) * 255).astype(np.uint8)
            cv2.imwrite(png_path, cv2.cvtColor(img_png, cv2.COLOR_RGB2BGR))
            saved_files[aov_name]["png"] = png_path

        # Save irradiance as EXR (linear color space for ST-RoomNet V3)
        elif aov_name == "irradiance":
            # INVERT: Model outputs bright=shadow, we need bright=light
            img_inverted = 1.0 - img_np

            # Save NPY (gamma-corrected, inverted)
            npy_path = os.path.join(output_dir, f"{filename_base}_irradiance_npy.npy")
            np.save(npy_path, img_inverted.astype(np.float32))
            saved_files[aov_name]["npy"] = npy_path

            # Reverse gamma correction to get linear RGB: linear = gamma^2.2
            img_linear = img_inverted ** 2.2
            exr_path = os.path.join(output_dir, f"{filename_base}_irradiance.exr")
            cv2.imwrite(exr_path, cv2.cvtColor(img_linear.astype(np.float32), cv2.COLOR_RGB2BGR))
            saved_files[aov_name]["exr"] = exr_path

            # Also save PNG preview (gamma-corrected for visual verification)
            png_path = os.path.join(output_dir, f"{filename_base}_irradiance_preview.png")
            img_png = (img_inverted * 255).astype(np.uint8)
            cv2.imwrite(png_path, cv2.cvtColor(img_png, cv2.COLOR_RGB2BGR))
            saved_files[aov_name]["png"] = png_path

    return saved_files

print("✓ AOV save helper function defined")

✓ AOV save helper function defined


---
## Cell 7: Processing Callback with Session Tracking
**Editable:** Main inference function. Modify parameters or add debug prints here.

In [17]:
# Initialize session state tracking (global dictionary)
processed_samples = {}

def process_image_callback(photo, seed, inference_step, num_samples, image_identifier):
    """
    Process a single image through RGB→X pipeline.

    Args:
        photo: Gradio File object or file path
        seed: Random seed for reproducibility
        inference_step: Number of DDIM denoising steps
        num_samples: Number of samples to generate (only first is saved)
        image_identifier: Unique identifier for session tracking

    Returns:
        (gallery_list, individual_zip_path, error_message)
    """
    try:
        # Get file path
        if hasattr(photo, 'name'):
            photo_path = photo.name
        else:
            photo_path = photo

        # Set random seed
        generator = torch.Generator(device="cuda").manual_seed(seed)

        # Load image based on format
        if photo_path.endswith(".exr"):
            photo_tensor = load_exr_image(photo_path, tonemaping=True, clamp=True).to("cuda")
        elif photo_path.endswith((".png", ".jpg", ".jpeg")):
            photo_tensor = load_ldr_image(photo_path, from_srgb=True).to("cuda")
        else:
            return [], None, f"Unsupported file format: {photo_path}"

        # Resize image to max 1000px with dimensions divisible by 8
        old_height = photo_tensor.shape[1]
        old_width = photo_tensor.shape[2]
        new_height = old_height
        new_width = old_width
        ratio = old_height / old_width
        max_side = 1000

        if old_height > old_width:
            new_height = max_side
            new_width = int(new_height / ratio)
        else:
            new_width = max_side
            new_height = int(new_width * ratio)

        # Ensure dimensions divisible by 8 (VAE requirement)
        if new_width % 8 != 0 or new_height % 8 != 0:
            new_width = new_width // 8 * 8
            new_height = new_height // 8 * 8

        photo_tensor = torchvision.transforms.Resize((new_height, new_width))(photo_tensor)

        # Define AOVs and prompts (frozen - do not modify)
        required_aovs = ["normal", "irradiance"]
        prompts = {
            "normal": "Camera-space Normal",
            "irradiance": "Irradiance (diffuse lighting)",
        }

        return_list = []
        aov_images = {}  # Store AOV images for download

        # Generate AOVs
        for i in range(num_samples):
            for aov_name in required_aovs:
                prompt = prompts[aov_name]

                # Get numpy array directly from pipeline
                generated_image_np = pipe(
                    prompt=prompt,
                    photo=photo_tensor,
                    num_inference_steps=inference_step,
                    height=new_height,
                    width=new_width,
                    generator=generator,
                    required_aovs=[aov_name],
                    output_type="np",  # Get numpy arrays directly (H, W, C)
                ).images[0][0]

                # Resize back to original dimensions
                generated_image_np = cv2.resize(
                    generated_image_np,
                    (old_width, old_height),
                    interpolation=cv2.INTER_LINEAR
                )

                # Store first sample for download
                if i == 0:
                    aov_images[aov_name] = generated_image_np

                # Convert to tensor for Gradio gallery display (C, H, W)
                display_tensor = torch.from_numpy(generated_image_np.transpose(2, 0, 1))
                return_list.append((display_tensor, f"Generated {aov_name} {i}"))

        # Create white wall × shadow visualization (debug)
        if "irradiance" in aov_images:
            shadow_np = aov_images["irradiance"]
            shadow_inverted = 1.0 - shadow_np
            white_surface = np.ones_like(shadow_inverted)
            shaded_white = white_surface * shadow_inverted
            shaded_white_tensor = torch.from_numpy(shaded_white.transpose(2, 0, 1))
            return_list.append((shaded_white_tensor, "White Wall × Shadow (Debug)"))

        # Save AOV files
        saved_files = save_aov_images(aov_images, os.path.basename(photo_path))

        # Create individual ZIP
        filename_base = os.path.splitext(os.path.basename(photo_path))[0]
        zip_path = create_hierarchical_zip(photo_path, saved_files, f"{filename_base}_output.zip")

        # Update session state
        processed_samples[image_identifier] = {
            "files": saved_files,
            "zip": zip_path,
            "input_path": photo_path,
            "success": True
        }

        return return_list, zip_path, ""

    except Exception as e:
        error_msg = f"Error processing image: {str(e)}"
        processed_samples[image_identifier] = {
            "success": False,
            "error": error_msg
        }
        return [], None, error_msg

print("✓ Processing callback defined")

✓ Processing callback defined


---
## Cell 8: Gradio Interface with Dynamic Sample Loading
**Editable:** Modify UI layout or add components here.

In [18]:
# Discover sample images dynamically
sample_images = sorted(
    glob_module.glob("sample_images/*.jpg") +
    glob_module.glob("sample_images/*.jpeg") +
    glob_module.glob("sample_images/*.png")
)

print(f"Found {len(sample_images)} sample images: {[os.path.basename(s) for s in sample_images]}")

# Create Gradio Blocks interface
demo = gr.Blocks(title="RGB→X: Normal & Irradiance Extraction")

with demo:
    gr.Markdown("# RGB→X Model: Normal Maps & Irradiance (Shadow) Extraction")
    gr.Markdown("Extract normal maps and shadow layers from room photos for ST-RoomNet V3 wallpaper visualization.")

    # ===== SECTION 1: Custom Upload =====
    gr.Markdown("## 1. Custom Image Upload")

    with gr.Row():
        # Left column: Input
        with gr.Column():
            gr.Markdown("### Input")
            custom_photo = gr.File(label="Upload Image", file_types=[".png", ".jpg", ".jpeg", ".exr"])

            gr.Markdown("### Parameters")
            custom_run_button = gr.Button("Run", variant="primary")

            with gr.Accordion("Advanced Options", open=False):
                custom_seed = gr.Slider(
                    label="Seed",
                    minimum=-1,
                    maximum=2147483647,
                    step=1,
                    value=42
                )
                custom_inference_step = gr.Slider(
                    label="Inference Steps",
                    minimum=1,
                    maximum=100,
                    step=1,
                    value=50
                )
                custom_num_samples = gr.Slider(
                    label="Number of Samples",
                    minimum=1,
                    maximum=5,
                    step=1,
                    value=1
                )

        # Right column: Output
        with gr.Column():
            gr.Markdown("### Output")
            custom_gallery = gr.Gallery(label="Generated AOVs", columns=3, height="auto")
            custom_download = gr.File(label="Download ZIP (Hierarchical Structure)")
            custom_error = gr.Textbox(label="Status", interactive=False)

    # Connect custom upload callback
    custom_run_button.click(
        fn=lambda photo, seed, steps, samples: process_image_callback(
            photo, seed, steps, samples, "custom_upload"
        ),
        inputs=[custom_photo, custom_seed, custom_inference_step, custom_num_samples],
        outputs=[custom_gallery, custom_download, custom_error],
        queue=True
    )

    gr.Markdown("---")

    # ===== SECTION 2: Dynamic Sample Images =====
    gr.Markdown(f"## 2. Sample Images ({len(sample_images)} available)")

    # Store components for batch processing
    sample_components = []

    for idx, sample_path in enumerate(sample_images):
        sample_name = os.path.basename(sample_path)
        sample_id = f"sample_{idx}"

        gr.Markdown(f"### Sample {idx + 1}: {sample_name}")

        with gr.Row():
            # Left column: Preloaded image and controls
            with gr.Column():
                sample_image = gr.Image(
                    value=sample_path,
                    label="Input Image",
                    type="filepath",
                    interactive=False
                )

                sample_run_button = gr.Button(f"Run Sample {idx + 1}", variant="primary")

                with gr.Accordion("Parameters", open=False):
                    sample_seed = gr.Slider(
                        label="Seed",
                        minimum=-1,
                        maximum=2147483647,
                        step=1,
                        value=42
                    )
                    sample_inference_step = gr.Slider(
                        label="Inference Steps",
                        minimum=1,
                        maximum=100,
                        step=1,
                        value=50
                    )
                    sample_num_samples = gr.Slider(
                        label="Number of Samples",
                        minimum=1,
                        maximum=5,
                        step=1,
                        value=1
                    )

            # Right column: Output
            with gr.Column():
                sample_gallery = gr.Gallery(label="Output", columns=3, height="auto")
                sample_download = gr.File(label="Download ZIP")
                sample_error = gr.Textbox(label="Status", interactive=False)

        # Connect sample callback
        sample_run_button.click(
            fn=lambda path=sample_path, seed=sample_seed, steps=sample_inference_step, samples=sample_num_samples, sid=sample_id: process_image_callback(
                path, seed.value if hasattr(seed, 'value') else seed,
                steps.value if hasattr(steps, 'value') else steps,
                samples.value if hasattr(samples, 'value') else samples,
                sid
            ),
            inputs=[sample_seed, sample_inference_step, sample_num_samples],
            outputs=[sample_gallery, sample_download, sample_error],
            queue=True
        )

        # Store for batch processing
        sample_components.append({
            "path": sample_path,
            "id": sample_id,
            "gallery": sample_gallery,
            "download": sample_download,
            "error": sample_error
        })

    gr.Markdown("---")

    # ===== SECTION 3: Batch Controls =====
    gr.Markdown("## 3. Batch Processing")

    with gr.Row():
        batch_process_button = gr.Button("Process All Samples", variant="secondary", size="lg")
        download_all_button = gr.Button("Download All (ZIP)", variant="primary", size="lg")

    batch_status = gr.Textbox(label="Batch Status", interactive=False)
    download_all_file = gr.File(label="Master ZIP Download")

print(f"✓ Gradio interface created with {len(sample_images)} sample images")

Found 12 sample images: ['bed_back_bedroom.jpg', 'complex_living.jpg', 'fluted_walls_living.jpg', 'living_simple2.jpg', 'living_small_walls.jpeg', 'no_walls_living.jpg', 'obstacle_course_living.jpg', 'perspective_living.jpg', 'room1.jpeg', 'simple_living.jpg', 'simple_living_2.jpeg', 'simple_living_3.jpg']
✓ Gradio interface created with 12 sample images


---
## Cell 9: Batch Processing Callback
**Editable:** Modify batch processing logic here.

In [19]:
def process_all_samples_callback():
    """
    Process all sample images sequentially.

    Returns:
        Status message
    """
    total = len(sample_images)
    successful = 0
    failed = 0

    status_msg = f"Processing {total} samples...\n\n"

    for idx, sample_path in enumerate(sample_images):
        sample_name = os.path.basename(sample_path)
        sample_id = f"sample_{idx}"

        try:
            status_msg += f"[{idx + 1}/{total}] Processing {sample_name}... "

            # Process with default parameters
            gallery, zip_file, error = process_image_callback(
                sample_path,
                seed=42,
                inference_step=50,
                num_samples=1,
                image_identifier=sample_id
            )

            if error:
                status_msg += f"FAILED\n  Error: {error}\n"
                failed += 1
            else:
                status_msg += "SUCCESS\n"
                successful += 1

        except Exception as e:
            status_msg += f"FAILED\n  Error: {str(e)}\n"
            failed += 1

    status_msg += f"\n{'='*50}\n"
    status_msg += f"Batch processing complete!\n"
    status_msg += f"Successful: {successful}/{total}\n"
    status_msg += f"Failed: {failed}/{total}\n"
    status_msg += f"{'='*50}"

    return status_msg

# Connect batch processing button
batch_process_button.click(
    fn=process_all_samples_callback,
    inputs=[],
    outputs=[batch_status],
    queue=True
)

print("✓ Batch processing callback defined and connected")

AttributeError: Cannot call click outside of a gradio.Blocks context.

---
## Cell 10: Download All Callback
**Editable:** Modify master ZIP creation logic here.

In [20]:
def download_all_callback():
    """
    Create master ZIP with all successfully processed samples.

    Returns:
        (master_zip_path, status_message)
    """
    # Filter for successful runs only
    successful_samples = {
        k: v for k, v in processed_samples.items()
        if v.get("success", False)
    }

    if not successful_samples:
        return None, "No successfully processed samples to download. Run 'Process All Samples' first."

    # Create master ZIP
    master_zip_path = "/tmp/all_samples_output.zip"

    with zipfile.ZipFile(master_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for sample_id, sample_data in successful_samples.items():
            input_path = sample_data["input_path"]
            input_filename = os.path.basename(input_path)
            filename_base = os.path.splitext(input_filename)[0]

            # Add input image
            zipf.write(input_path, arcname=f"input_images/{input_filename}")

            # Add AOV files
            aov_files = sample_data["files"]
            for aov_name, file_dict in aov_files.items():
                for file_type, file_path in file_dict.items():
                    if file_path and os.path.exists(file_path):
                        # Determine folder and filename
                        if aov_name == "normal":
                            folder = "normal_model"
                            if file_type == "npy":
                                arc_name = f"{folder}/{filename_base}_normal_npy.npy"
                            else:  # png
                                arc_name = f"{folder}/{filename_base}_normal_preview.png"

                        elif aov_name == "irradiance":
                            folder = "irradiance_model"
                            if file_type == "npy":
                                arc_name = f"{folder}/{filename_base}_irradiance_npy.npy"
                            elif file_type == "exr":
                                arc_name = f"{folder}/{filename_base}_irradiance.exr"
                            else:  # png
                                arc_name = f"{folder}/{filename_base}_irradiance_preview.png"

                        zipf.write(file_path, arcname=arc_name)

    # Calculate success rate
    total_processed = len(processed_samples)
    successful_count = len(successful_samples)

    status_msg = f"✓ Master ZIP created successfully!\n"
    status_msg += f"Included: {successful_count}/{total_processed} processed samples\n"
    status_msg += f"Structure: input_images/ + normal_model/ + irradiance_model/"

    return master_zip_path, status_msg

# Connect download all button
download_all_button.click(
    fn=download_all_callback,
    inputs=[],
    outputs=[download_all_file, batch_status],
    queue=True
)

print("✓ Download all callback defined and connected")

AttributeError: Cannot call click outside of a gradio.Blocks context.

---
## Cell 11: Launch Gradio Demo
**Run this to start the interface:** Creates public URL accessible from any browser.

In [ ]:
# Configure and launch demo
demo.queue(max_size=10)
demo.launch(share=True, debug=True)

print("\n" + "="*60)
print("✓ Gradio demo launched!")
print("Access the public URL above (https://xxx.gradio.live)")
print("\nTo modify the interface:")
print("1. Edit Cells 7-10 (processing logic)")
print("2. Re-run edited cells")
print("3. Re-run Cell 8 (interface) to refresh UI")
print("4. Re-run this cell (Cell 11) to launch with changes")
print("\nModel stays loaded - NO need to re-run Cell 4!")
print("="*60)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://e5587bd46387830206.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
